## Imports

In [8]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Detect environment and set data directory
if Path("/kaggle/input").exists():
    # Kaggle environment
    DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e9")
else:
    # Local environment
    PROJECT_ROOT = Path.cwd().parent
    DATA_DIR = PROJECT_ROOT / "data" / "raw"


print(f"Data directory: {DATA_DIR}")

Data directory: c:\Users\Darío\Desktop\KAGGLE\Predicting_Electric_Vehicle_Purchases\data\raw


## Data loading

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

Train shape: (668665, 15)
Test shape: (286571, 14)
Sample submission shape: (286571, 2)


In [4]:
target = "Will_Buy_EV"

X = train.drop(columns=[target, "id"])
y = train[target]

X_test = test.drop(columns=["id"])

## Splits

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Numerical and cathegorical features

In [6]:
numeric_features = X_train.select_dtypes(include="number").columns

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns

print("Numeric features:")
print(list(numeric_features))

print("\nCategorical features:")
print(list(categorical_features))

Numeric features:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

Categorical features:
['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


C:\Users\Darío\AppData\Local\Temp\ipykernel_25936\3764160197.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


The standard scaler is not required for our Random Forest model, but we still need the One Hot Enconder for the categorical columns

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [10]:
model.fit(X_train, y_train)

y_val_proba = model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, y_val_proba)

print(f"Validation ROC-AUC: {roc_auc:.4f}")

Validation ROC-AUC: 0.9324


In [12]:
model.fit(X, y)

test_proba = model.predict_proba(test.drop(columns=["id"]))[:, 1]

submission = sample_submission.copy()

submission["Will_Buy_EV"] = test_proba

display(submission.head())

,id,Will_Buy_EV
0,668665,0.00
1,668666,0.05
2,668667,0.02
3,668668,0.00
4,668669,0.01


In [14]:
OUTPUT_DIR = Path("../data/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission_path = OUTPUT_DIR / "baseline_random_forest.csv"

submission.to_csv(submission_path, index=False)

print("Saved:", submission_path.resolve())

Saved: C:\Users\Darío\Desktop\KAGGLE\Predicting_Electric_Vehicle_Purchases\data\outputs\baseline_random_forest.csv
